In [ ]:
%%writefile fmm.cpp
#include <cstdio>    
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
const float INF = 1e30f;

float solve_quadratic(float a, float b, float F, float h)
{
    float Fh = F * h;
    if (fabsf(a - b) <= Fh)                                   
        return 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));
    return fminf(a, b) + Fh;                                  
}

float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int i, int j, int N, float h)
{
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[i * N + (j - 1)] : INF;
    float right = (j < N - 1) ? u[i * N + (j + 1)] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[(i - 1) * N + j] : INF;
    float up    = (i < N - 1) ? u[(i + 1) * N + j] : INF;
    // Per arrivare devi scorrere come un indice globale: u[i][j] = u[i * N + j]!

    float a = fminf(left, right);
    float b = fminf(down, up);

    int idx = i * N + j;
    float q = solve_quadratic(a, b, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ij, u)
}

int main()
{
    
    const int   N = 5;                      // nodi per lato (dispari: la sorgente cade su un nodo)
    const float h = 2.0f / (N - 1);         // spaziatura del grigliato
    typedef std::pair<float, int> Entry;    // coppia di float e int: (u, indice), nome Entry tramite typedef

    
    std::vector<float> u(N * N, INF);       // array di float NxN, inizializzato a INF
    std::vector<float> F(N * N, 1.0f);      // array di float NxN, inizializzato a 1.0f

    // Sorgente puntiforme nell'origine
    const int si = N / 2, sj = N / 2;       // riga e colonna del nodo centrale
    u[si * N + sj] = 0.0f;  // equivalente u[si][sj] = 0.0f

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    float valori[7] = { 9, 3, 15, 1, 7, 6, 5 };            // i valori della fig. 2.8, in ordine sparso
    for (int k = 0; k < 7; ++k)
        heap.push(Entry(valori[k], k));                   // inserimento: O(log n)
    printf("radice (minimo): u = %.0f\n", heap.top().first);
    printf("estrazione:");
    while (!heap.empty()) {
        printf(" %.0f", heap.top().first);                // Trial <- radice del min-heap
        heap.pop();                                       // rimuovi Trial
    }
    printf("\n");
}
